# Cross-Currency Basis: USD Bond for a EUR Investor

A EUR-based investor buying a US Treasury must hedge the FX risk. The cost of that hedge depends on the **XCCY basis** — the spread between market FX forwards and CIP-theoretical forwards.

**What we build:**
1. USD curve (SOFR/OIS) + EUR curve (ESTR/OIS)
2. XCCY basis from market FX forwards (EURUSD)
3. Basis-adjusted EUR discount curve
4. FX-hedged yield of US Treasuries for a EUR investor
5. Carry pickup vs Bunds — is the trade worth it after hedging?

Market data modelled on **Jul 2024** — negative XCCY basis (USD funding premium).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.bond import FixedRateBond
from pricebook.bootstrap import bootstrap
from pricebook.bond_desk import fit_curve_from_bonds
from pricebook.xccy_bond import fx_hedged_yield, cross_currency_pickup, breakeven_fx_move
from pricebook.xccy_basis import implied_basis_spread, bootstrap_basis_curve
from pricebook.day_count import DayCountConvention
from pricebook.schedule import Frequency
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 7, 15)
print(f"Valuation date: {REF}")

## 1. Build both OIS curves

In [ ]:
# USD SOFR/OIS curve
usd_deposits = [
    (REF + relativedelta(months=1), 0.0533),
    (REF + relativedelta(months=3), 0.0531),
    (REF + relativedelta(months=6), 0.0518),
]
usd_swaps = [
    (REF + relativedelta(years=1),  0.0495),
    (REF + relativedelta(years=2),  0.0460),
    (REF + relativedelta(years=3),  0.0440),
    (REF + relativedelta(years=5),  0.0415),
    (REF + relativedelta(years=7),  0.0405),
    (REF + relativedelta(years=10), 0.0395),
    (REF + relativedelta(years=20), 0.0410),
    (REF + relativedelta(years=30), 0.0420),
]
usd_curve = bootstrap(REF, usd_deposits, usd_swaps)

# EUR ESTR/OIS curve
eur_deposits = [
    (REF + relativedelta(months=1), 0.0370),
    (REF + relativedelta(months=3), 0.0368),
    (REF + relativedelta(months=6), 0.0355),
]
eur_swaps = [
    (REF + relativedelta(years=1),  0.0340),
    (REF + relativedelta(years=2),  0.0310),
    (REF + relativedelta(years=3),  0.0295),
    (REF + relativedelta(years=5),  0.0280),
    (REF + relativedelta(years=7),  0.0275),
    (REF + relativedelta(years=10), 0.0275),
    (REF + relativedelta(years=20), 0.0275),
    (REF + relativedelta(years=30), 0.0260),
]
eur_curve = bootstrap(REF, eur_deposits, eur_swaps)

print(f"{'Tenor':>6}  {'USD SOFR':>9}  {'EUR ESTR':>9}  {'Rate Diff':>9}")
print(f"{'─'*6}  {'─'*9}  {'─'*9}  {'─'*9}")
for t in [1, 2, 3, 5, 7, 10, 20, 30]:
    d = REF + relativedelta(years=t)
    usd_z = usd_curve.zero_rate(d)
    eur_z = eur_curve.zero_rate(d)
    print(f"{t:>5}Y  {usd_z*100:>8.2f}%  {eur_z*100:>8.2f}%  {(usd_z-eur_z)*1e4:>+8.1f}bp")

## 2. XCCY Basis from FX forwards

CIP says: `F = S × df_EUR / df_USD`. In practice, market forwards differ due to the cross-currency basis.

Jul 2024: EURUSD spot ≈ 1.09, basis ≈ -10 to -25bp (USD funding premium).

In [ ]:
SPOT = 1.0900  # EURUSD spot

# Market FX forwards (EURUSD) — include basis
# CIP-theoretical forward + basis adjustment
forward_tenors = [1, 2, 3, 5, 7, 10, 20, 30]
basis_bp = [-12, -15, -18, -22, -20, -18, -14, -10]  # typical Jul 2024 levels

market_forwards = []
theoretical_forwards = []

print(f"{'Tenor':>5}  {'CIP Fwd':>9}  {'Mkt Fwd':>9}  {'Basis':>8}")
print(f"{'─'*5}  {'─'*9}  {'─'*9}  {'─'*8}")

for t, b in zip(forward_tenors, basis_bp):
    d = REF + relativedelta(years=t)
    df_eur = eur_curve.df(d)
    df_usd = usd_curve.df(d)
    
    # CIP theoretical: F = S × df_EUR / df_USD
    f_cip = SPOT * df_eur / df_usd
    theoretical_forwards.append(f_cip)
    
    # Market forward includes basis: adjust EUR df by basis
    # A negative basis means USD is more expensive to fund
    # F_market = S × df_EUR_adj / df_USD where df_EUR_adj uses (eur_zero + basis)
    import math
    tau = t
    eur_z = eur_curve.zero_rate(d)
    df_eur_adj = math.exp(-(eur_z + b / 1e4) * tau)
    f_mkt = SPOT * df_eur_adj / df_usd
    market_forwards.append((d, f_mkt))
    
    print(f"{t:>4}Y  {f_cip:>9.4f}  {f_mkt:>9.4f}  {b:>+7d}bp")

In [ ]:
# Bootstrap basis-adjusted EUR curve from market forwards
eur_basis_curve = bootstrap_basis_curve(
    REF, SPOT, market_forwards, eur_curve, usd_curve
)

# Verify: implied basis at each tenor
print("Implied XCCY basis from market forwards:")
print(f"{'Tenor':>5}  {'EUR OIS':>8}  {'EUR+Basis':>9}  {'Basis':>8}")
print(f"{'─'*5}  {'─'*8}  {'─'*9}  {'─'*8}")

basis_values = []
for (d, f_mkt), t in zip(market_forwards, forward_tenors):
    basis = implied_basis_spread(SPOT, d, f_mkt, eur_curve, usd_curve)
    eur_z = eur_curve.zero_rate(d)
    eur_adj_z = eur_basis_curve.zero_rate(d)
    basis_values.append(basis * 1e4)
    print(f"{t:>4}Y  {eur_z*100:>7.2f}%  {eur_adj_z*100:>8.2f}%  {basis*1e4:>+7.1f}bp")

In [ ]:
# Plot: XCCY basis term structure
with apply_theme():
    fig, (ax1, ax2) = create_figure(2)

    # Basis term structure
    ax1.plot(forward_tenors, basis_values, 'o-', linewidth=2, color='tab:red')
    ax1.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("XCCY Basis (bp)")
    ax1.set_title("EURUSD Cross-Currency Basis Term Structure")
    ax1.fill_between(forward_tenors, basis_values, 0, alpha=0.2, color='tab:red')

    # EUR curves: OIS vs basis-adjusted
    tenors_plot = np.linspace(0.5, 30, 60)
    eur_z_plot = []
    eur_adj_plot = []
    for t in tenors_plot:
        d = REF + relativedelta(days=int(t * 365.25))
        eur_z_plot.append(eur_curve.zero_rate(d) * 100)
        eur_adj_plot.append(eur_basis_curve.zero_rate(d) * 100)

    ax2.plot(tenors_plot, eur_z_plot, '-', linewidth=2, label="EUR OIS")
    ax2.plot(tenors_plot, eur_adj_plot, '--', linewidth=2, label="EUR + XCCY basis")
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("Zero Rate (%)")
    ax2.set_title("EUR Curve: OIS vs Basis-Adjusted")
    ax2.legend(fontsize=9)

    fig.tight_layout()

## 3. FX-hedged yield of US Treasuries

A EUR investor buying a 10Y UST at 4.25% yield, hedging EURUSD:
- **Unhedged yield** = 4.25% (in USD)
- **FX hedge cost** = USD rate - EUR rate + basis ≈ 1.5-1.6%
- **FX-hedged yield** = 4.25% - 1.5% ≈ 2.7% (in EUR terms)

Compare to Bund 10Y at 2.75% — is the pickup worth the basis risk?

In [ ]:
# US Treasury on-the-run universe
ust_bonds = [
    ("2Y",  0.0475, date(2024, 6, 30), date(2026, 6, 30),  99.750),
    ("5Y",  0.0425, date(2024, 4, 30), date(2029, 4, 30),  99.000),
    ("10Y", 0.0425, date(2024, 2, 15), date(2034, 2, 15), 102.500),
    ("30Y", 0.0437, date(2024, 2, 15), date(2054, 2, 15),  96.000),
]

# Bund benchmarks for comparison
bund_yields = {"2Y": 0.0290, "5Y": 0.0275, "10Y": 0.0275, "30Y": 0.0262}

print("FX-Hedged UST Yields for EUR Investor")
print(f"{'Tenor':>5}  {'UST YTM':>8}  {'Hedge Cost':>10}  {'Hedged':>8}  {'Bund':>7}  {'Pickup':>8}  {'BE FX':>8}")
print(f"{'─'*5}  {'─'*8}  {'─'*10}  {'─'*8}  {'─'*7}  {'─'*8}  {'─'*8}")

hedged_yields = []
ust_yields = []
bund_y_list = []
tenor_labels = []

for label, cpn, issue, mat, px in ust_bonds:
    bond = FixedRateBond.treasury_note(issue, mat, cpn)
    settle = bond.settlement_date(REF)
    ai = bond.accrued_interest(settle)
    ytm = bond.yield_to_maturity(px + ai, settle)
    T = (mat - REF).days / 365.25

    # Rate differential at this tenor
    d = REF + relativedelta(years=int(T))
    usd_z = usd_curve.zero_rate(d)
    eur_z = eur_curve.zero_rate(d)

    # EUR investor: domestic=EUR, foreign=USD
    bund_y = bund_yields.get(label, 0.0275)
    result = cross_currency_pickup(ytm, bund_y, eur_z, usd_z)

    # Breakeven FX move (unhedged)
    be = breakeven_fx_move(ytm, bund_y, T)

    hedged_yields.append(result.fx_hedged_yield * 100)
    ust_yields.append(ytm * 100)
    bund_y_list.append(bund_y * 100)
    tenor_labels.append(label)

    print(f"{label:>5}  {ytm*100:>7.2f}%  {result.fx_hedging_cost*100:>+9.2f}%  "
          f"{result.fx_hedged_yield*100:>7.2f}%  {bund_y*100:>6.2f}%  "
          f"{result.carry_pickup*1e4:>+7.0f}bp  {be*100:>+7.2f}%")

In [ ]:
# Plot: yield comparison
with apply_theme():
    fig, (ax1, ax2) = create_figure(2)

    x = np.arange(len(tenor_labels))
    width = 0.3
    ax1.bar(x - width, ust_yields, width, label='UST (unhedged, USD)', alpha=0.8)
    ax1.bar(x, hedged_yields, width, label='UST (FX-hedged, EUR)', alpha=0.8)
    ax1.bar(x + width, bund_y_list, width, label='Bund (EUR)', alpha=0.8)
    ax1.set_ylabel('Yield (%)')
    ax1.set_title('UST vs Bund: Unhedged, Hedged, and Domestic')
    ax1.set_xticks(x)
    ax1.set_xticklabels(tenor_labels)
    ax1.legend(fontsize=8)

    # Pickup after hedging
    pickup = [h - b for h, b in zip(hedged_yields, bund_y_list)]
    colors = ['tab:green' if p > 0 else 'tab:red' for p in pickup]
    ax2.bar(tenor_labels, pickup, color=colors, alpha=0.8)
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_ylabel('Pickup (pp)')
    ax2.set_title('FX-Hedged UST Pickup over Bund')

    fig.tight_layout()

## 4. Sensitivity to XCCY basis changes

What happens to the hedged yield when the basis moves?

In [ ]:
# Take 10Y UST as example
bond_10y = FixedRateBond.treasury_note(date(2024, 2, 15), date(2034, 2, 15), 0.0425)
settle = bond_10y.settlement_date(REF)
ai = bond_10y.accrued_interest(settle)
ytm_10y = bond_10y.yield_to_maturity(102.500 + ai, settle)

d10 = REF + relativedelta(years=10)
usd_z10 = usd_curve.zero_rate(d10)
eur_z10 = eur_curve.zero_rate(d10)
bund_10y = 0.0275

print("10Y UST Hedged Yield Sensitivity to XCCY Basis")
print(f"{'Basis':>8}  {'Hedge Cost':>10}  {'Hedged YTM':>10}  {'Pickup':>8}")
print(f"{'─'*8}  {'─'*10}  {'─'*10}  {'─'*8}")

basis_scenarios = [-40, -30, -20, -18, -10, 0, +10, +20]
hedged_by_basis = []

for b in basis_scenarios:
    # Basis affects the effective EUR funding rate
    # EUR investor: domestic=EUR (adjusted by basis), foreign=USD
    eur_eff = eur_z10 + b / 1e4
    result = cross_currency_pickup(ytm_10y, bund_10y, eur_eff, usd_z10)
    hedged_by_basis.append(result.fx_hedged_yield * 100)
    marker = " <-- current" if b == -18 else ""
    print(f"{b:>+7d}bp  {result.fx_hedging_cost*100:>+9.2f}%  "
          f"{result.fx_hedged_yield*100:>9.2f}%  {result.carry_pickup*1e4:>+7.0f}bp{marker}")

# Plot
with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(basis_scenarios, hedged_by_basis, 'o-', linewidth=2)
    ax.axhline(bund_10y * 100, color='tab:orange', linestyle='--',
               linewidth=1.5, label=f'Bund 10Y ({bund_10y*100:.2f}%)')
    ax.axvline(-18, color='tab:gray', linestyle=':', linewidth=1, label='Current basis (-18bp)')
    ax.set_xlabel('XCCY Basis (bp)')
    ax.set_ylabel('FX-Hedged UST 10Y Yield (%)')
    ax.set_title('Hedged Yield Sensitivity to XCCY Basis')
    ax.legend(fontsize=9)
    fig.tight_layout()

## Summary

| Component | What it does |
|-----------|-------------|
| **CIP (covered interest parity)** | `F = S × df_dom / df_for` — the no-arbitrage forward |
| **XCCY basis** | Market deviation from CIP — reflects USD funding premium |
| **FX hedge cost** | `USD rate - EUR rate` ≈ 1.2-1.6% (Jul 2024) |
| **FX-hedged yield** | UST yield - hedge cost — what a EUR investor actually earns |
| **Pickup** | Hedged yield - Bund yield — is it worth crossing currencies? |

**Key insights:**
- In Jul 2024, the **rate differential** (SOFR ~5% vs ESTR ~3.7%) makes FX hedging expensive
- The **negative XCCY basis** (-10 to -25bp) makes it even more expensive (USD is "special")
- After hedging, the **pickup over Bunds** is often small or negative for short tenors
- The trade only works at the **long end** where the UST term premium exceeds the hedge cost
- A **widening basis** (more negative) kills the trade; **tightening** (toward zero) helps
- **Breakeven FX move** shows how much EUR/USD can move before an unhedged position loses